In [ ]:
%cd /content
!rm -rf epistasis-budget
!git clone -b audit/scientific-hardening https://github.com/VivienP/epistasis-budget.git
%cd /content/epistasis-budget
!pip install -e . -q
!pip install -q torch transformers huggingface_hub
import sys
sys.path.insert(0, "src")
import epibudget
print("OK :", epibudget.__file__)
import torch
print("CUDA :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

/content
Cloning into 'epistasis-budget'...
remote: Enumerating objects: 552, done.
remote: Counting objects: 100% (552/552), done.
remote: Compressing objects: 100% (296/296), done.
remote: Total 552 (delta 325), reused 431 (delta 206), pack-reused 0 (from 0)
Receiving objects: 100% (552/552), 470.92 KiB | 10.02 MiB/s, done.
Resolving deltas: 100% (325/325), done.
/content/epistasis-budget
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for epibudget (pyproject.toml) ... done
OK : /content/epistasis-budget/src/epibudget/__init__.py
CUDA : True Tesla T4


In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download
os.makedirs("data/proteingym", exist_ok=True)
src = hf_hub_download(repo_id="SeprotHub/Dataset-TrpB_fitness_landsacpe",
                      filename="dataset.csv", repo_type="dataset")
shutil.copy(src, "data/proteingym/trpb_johnston2024.csv")
!wc -l data/proteingym/trpb_johnston2024.csv   # expected: 160001

dataset.csv:   0%|          | 0.00/66.6M [00:00<?, ?B/s]

160001 data/proteingym/trpb_johnston2024.csv


In [ ]:
!cd /content/epistasis-budget && PYTHONPATH=src python -m epibudget.cli validate \
    --dataset trpb_johnston2024 --model esm2_t33_650M --device cuda \
    --scored-cache report/scored_trpb_650m_n0.jsonl \
    --alphabet ACDEFGHIKLMNPQRSTVWY --budgets 48,96,192 --seeds 20 \
    --n-perturbations 0 --batch-size 128 --out report/

validate trpb_johnston2024: scoring 29678 candidates 
(alphabet='ACDEFGHIKLMNPQRSTVWY') with esm2_t33_650M on cuda …
config.json: 100% 724/724 [00:00<00:00, 4.00MB/s]
tokenizer_config.json: 100% 95.0/95.0 [00:00<00:00, 554kB/s]
vocab.txt: 100% 93.0/93.0 [00:00<00:00, 830kB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 1.12MB/s]
model.safetensors: 100% 2.61G/2.61G [00:10<00:00, 253MB/s]
Loading weights: 100% 539/539 [00:00<00:00, 5500.18it/s]
Predicted Var[ε] = 0.0316  [PASS invariant #1]  tolerance=3.692e-14   
candidates=29678  alphabet='ACDEFGHIKLMNPQRSTVWY'  truth_terms=17709
Truth Var[ε] = 3.9302
               recovery — pairwise (Spearman / Pearson, coverage)               
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃   B ┃         info ┃     fitness ┃   structural ┃      random ┃     practice ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│  48 │ +0.177/+0.1… │ +0.132/+0.… │ +0.177/+0.1… │ -0.125/

In [ ]:
from google.colab import files
files.download("report/scored_trpb_650m_n0.jsonl")
files.download("report/scored_trpb_650m_n0.jsonl.meta.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>